In [4]:
# [Cell 0: 강제 클린 설치]
# 최신 파이토치 환경에서 PyG를 가장 안정적으로 불러오는 방식입니다.
!pip uninstall -y torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.3.0+cu121.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-2.3.0+cu121.html
!pip install torch-geometric

# 여기서 끝이 아닙니다!
import sys
# 혹시 모를 경로 충돌을 방지하기 위해 강제로 import 경로를 확인합니다.
import torch_geometric
print("✅ 설치 및 임포트 성공!")

Found existing installation: torch-geometric 2.7.0
Uninstalling torch-geometric-2.7.0:
  Successfully uninstalled torch-geometric-2.7.0


Looking in links: https://data.pyg.org/whl/torch-2.3.0+cu121.html
     ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
     --- ------------------------------------ 0.3/3.5 MB ? eta -:--:--
     ------------------ --------------------- 1.6/3.5 MB 4.4 MB/s eta 0:00:01
     ---------------------------------------- 3.5/3.5 MB 6.0 MB/s  0:00:01
Looking in links: https://data.pyg.org/whl/torch-2.3.0+cu121.html
     ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
     ---------------------------------------- 2.1/2.1 MB 11.5 MB/s  0:00:00
  Using cached torch_geometric-2.7.0-py3-none-any.whl.metadata (63 kB)
Using cached torch_geometric-2.7.0-py3-none-any.whl (1.3 MB)
✅ 설치 및 임포트 성공!


In [5]:
import os
import zipfile
import glob
from google.colab import drive

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 작업 폴더 생성
extract_path = '/content/ASD_project_data'
os.makedirs(extract_path, exist_ok=True)

# 3. 드라이브 내 zip 파일 탐색 및 압축 해제
# 드라이브 최상단에 zip 파일을 올려두셨다고 가정합니다.
# 파일이 다른 폴더에 있다면 아래 경로를 수정하세요.
zip_files = glob.glob('/content/drive/MyDrive/*.zip')

if not zip_files:
    print("🚨 에러: 구글 드라이브 최상단에서 .zip 파일을 찾을 수 없습니다!")
    print("💡 좌측 폴더 아이콘(📁)을 눌러 파일 경로를 확인해주세요.")
else:
    print(f"📦 총 {len(zip_files)}개의 압축 파일을 발견했습니다.")
    for zip_file in zip_files:
        print(f"  - {os.path.basename(zip_file)} 푸는 중...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
    print(f"🎉 데이터 준비 완료! 이제 아래의 전처리 코드를 실행하세요.")

ModuleNotFoundError: No module named 'google'

In [ ]:
import random
import os
import numpy as np
import torch

# 🌟 주피터든 코랩이든 무조건 똑같은 결과가 나오도록 '운(Seed)'을 통제합니다!
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # 멀티 GPU 사용 시 필요
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [ ]:
# ==========================================
# 🧠 [진짜 최종 마스터 코드] 피처 엔지니어링(4ch) & GNN 3대장 다이어트 고도화
# ==========================================
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import WeightedRandomSampler
from torch_geometric.nn import GraphConv, ChebConv, TransformerConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import degree # 💡 2단계: 차수 계산을 위한 모듈
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
# 💡 맨 앞에 GCNConv를 슬쩍 끼워 넣어줍니다!
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, GINConv, global_mean_pool, global_add_pool, global_max_pool, BatchNorm
# 🔥 코랩 T4 GPU 풀 가동
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n======================================")
print(f" 🔥 장치 상태: {DEVICE} (이름: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음'})")
print(f"======================================\n")

# 🌟 모든 랜덤성 제어 (동일 환경 재현 보장)
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# ==========================================
# 1. 3단계: 고도화 및 다이어트가 완료된 GNN 모델 설계도
# ==========================================

# ==========================================
# 1. 3단계: 고도화 및 다이어트가 완료된 GNN 모델 설계도 (Mean Pooling 롤백)
# ==========================================
class GCNModel(nn.Module):
    def __init__(self, in_ch, hidden=16, num_classes=2, dropout=0.1):
        super().__init__()
        self.c1 = GCNConv(in_ch, hidden)
        self.c2 = GCNConv(hidden, hidden)
        self.bn1 = BatchNorm(hidden)
        self.bn2 = BatchNorm(hidden)
        self.drop = nn.Dropout(dropout)
        self.clf = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), self.drop, nn.Linear(32, num_classes))

    def _encode(self, data):
        x, ei, ew, b = data.x, data.edge_index, data.edge_attr, data.batch
        # GCNConv는 1차원 엣지 가중치를 지원합니다.
        weight = ew.squeeze(-1) if ew is not None else None
        h1 = self.drop(F.relu(self.bn1(self.c1(x, ei, edge_weight=weight))))
        h2 = F.relu(self.bn2(self.c2(h1, ei, edge_weight=weight)))
        out = h1 + h2
        return global_mean_pool(out, b)

    def forward(self, data): return self.clf(self._encode(data))

class GATModel(nn.Module):
    def __init__(self, in_ch, hidden=16, heads=2, num_classes=2, dropout=0.1):
        super().__init__()
        # edge_dim=1을 추가하여 fMRI의 핵심인 엣지 가중치(상관계수)를 모델이 적극 반영하도록 설정
        self.c1 = GATConv(in_ch, hidden, heads=heads, dropout=dropout, concat=False, edge_dim=1)
        self.c2 = GATConv(hidden, hidden, heads=heads, dropout=dropout, concat=False, edge_dim=1)

        self.bn1 = BatchNorm(hidden)
        self.bn2 = BatchNorm(hidden)
        self.drop = nn.Dropout(dropout)

        # 💡 분류기 입력 차원 원래대로(hidden) 복구
        self.clf = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), self.drop, nn.Linear(32, num_classes))

    def _encode(self, data):
        x, ei, ew, b = data.x, data.edge_index, data.edge_attr, data.batch
        h1 = self.drop(F.elu(self.bn1(self.c1(x, ei, edge_attr=ew))))
        h2 = F.elu(self.bn2(self.c2(h1, ei, edge_attr=ew)))
        out = h1 + h2 # 잔차 연결

        # 💡 순수 global_mean_pool 로 복구
        return global_mean_pool(out, b)

    def forward(self, data): return self.clf(self._encode(data))


class GINModel(nn.Module):
    def __init__(self, in_ch, hidden=16, num_classes=2, dropout=0.1):
        super().__init__()
        def mlp(a,b): return nn.Sequential(nn.Linear(a,b), nn.BatchNorm1d(b), nn.ReLU(), nn.Linear(b,b))

        self.c1 = GINConv(mlp(in_ch, hidden), train_eps=True)
        self.c2 = GINConv(mlp(hidden, hidden), train_eps=True)
        self.drop = nn.Dropout(dropout)

        # 💡 기존 GIN은 입력 차원이 hidden*2 (global_add_pool 두 개 합침) 였습니다.
        self.clf = nn.Sequential(nn.Linear(hidden*2, 32), nn.ReLU(), self.drop, nn.Linear(32, num_classes))

    def _encode(self, data):
        x, ei, b = data.x, data.edge_index, data.batch
        h1 = self.drop(F.relu(self.c1(x, ei)))
        h2 = F.relu(self.c2(h1, ei))

        # 💡 원본 GIN 구조 복구
        return torch.cat([global_add_pool(h,b) for h in [h1, h2]], dim=1)

    def forward(self, data): return self.clf(self._encode(data))


class SAGEModel(nn.Module):
    def __init__(self, in_ch, hidden=16, num_classes=2, dropout=0.1):
        super().__init__()
        self.convs = nn.ModuleList([SAGEConv(in_ch, hidden), SAGEConv(hidden, hidden)])
        self.bns   = nn.ModuleList([BatchNorm(hidden)]*2)
        self.drop  = nn.Dropout(dropout)

        # 💡 원본 SAGE는 Mean + Max 였습니다. 그대로 둡니다.
        self.clf   = nn.Sequential(nn.Linear(hidden*2, 32), nn.ReLU(), self.drop, nn.Linear(32, num_classes))

    def _encode(self, data):
        x, ei, b = data.x, data.edge_index, data.batch
        h1 = self.drop(F.relu(self.bns[0](self.convs[0](x, ei))))
        h2 = F.relu(self.bns[1](self.convs[1](h1, ei)))
        out = h1 + h2 # 잔차 연결

        # 💡 원본 SAGE 구조 복구
        return torch.cat([global_mean_pool(out, b), global_max_pool(out, b)], dim=1)

    def forward(self, data): return self.clf(self._encode(data))

# ==========================================
# 🌟 [확장] 팀장님 오리지널 3개 GNN 추가
# ==========================================

class GraphConvModel(nn.Module):
    def __init__(self, in_ch, hidden=16, num_classes=2, dropout=0.1):
        super().__init__()
        self.c1 = GraphConv(in_ch, hidden)
        self.c2 = GraphConv(hidden, hidden)
        self.bn1 = BatchNorm(hidden)
        self.bn2 = BatchNorm(hidden)
        self.drop = nn.Dropout(dropout)
        self.clf = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), self.drop, nn.Linear(32, num_classes))

    def _encode(self, data):
        x, ei, ew, b = data.x, data.edge_index, data.edge_attr, data.batch
        weight = ew.squeeze(-1) if ew is not None else None
        h1 = self.drop(F.relu(self.bn1(self.c1(x, ei, edge_weight=weight))))
        h2 = F.relu(self.bn2(self.c2(h1, ei, edge_weight=weight)))
        out = h1 + h2
        return global_mean_pool(out, b)

    def forward(self, data): return self.clf(self._encode(data))

class ChebModel(nn.Module):
    def __init__(self, in_ch, hidden=16, num_classes=2, dropout=0.1):
        super().__init__()
        self.c1 = ChebConv(in_ch, hidden, K=2) # K: 홉(Hop) 수
        self.c2 = ChebConv(hidden, hidden, K=2)
        self.bn1 = BatchNorm(hidden)
        self.bn2 = BatchNorm(hidden)
        self.drop = nn.Dropout(dropout)
        self.clf = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), self.drop, nn.Linear(32, num_classes))

    def _encode(self, data):
        x, ei, ew, b = data.x, data.edge_index, data.edge_attr, data.batch
        weight = ew.squeeze(-1) if ew is not None else None
        h1 = self.drop(F.relu(self.bn1(self.c1(x, ei, edge_weight=weight))))
        h2 = F.relu(self.bn2(self.c2(h1, ei, edge_weight=weight)))
        out = h1 + h2
        return global_mean_pool(out, b)

    def forward(self, data): return self.clf(self._encode(data))

class TransformerModel(nn.Module):
    def __init__(self, in_ch, hidden=16, num_classes=2, dropout=0.1):
        super().__init__()
        self.c1 = TransformerConv(in_ch, hidden, heads=2, concat=False, edge_dim=1)
        self.c2 = TransformerConv(hidden, hidden, heads=2, concat=False, edge_dim=1)
        self.bn1 = BatchNorm(hidden)
        self.bn2 = BatchNorm(hidden)
        self.drop = nn.Dropout(dropout)
        self.clf = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), self.drop, nn.Linear(32, num_classes))

    def _encode(self, data):
        x, ei, ew, b = data.x, data.edge_index, data.edge_attr, data.batch
        h1 = self.drop(F.relu(self.bn1(self.c1(x, ei, edge_attr=ew))))
        h2 = F.relu(self.bn2(self.c2(h1, ei, edge_attr=ew)))
        out = h1 + h2
        return global_mean_pool(out, b)

    def forward(self, data): return self.clf(self._encode(data))

# 💡 수정된 모델 등록 함수 (자동 감지 모드)
# 💡 완벽한 7개 모델 풀세트 등록
def get_model_registry(n_rois):
    return [
        ("GCN", GCNModel, {"in_ch": n_rois}),
        ("GAT", GATModel, {"in_ch": n_rois}),
        ("GIN", GINModel, {"in_ch": n_rois}),
        ("GraphSAGE", SAGEModel, {"in_ch": n_rois}),
        ("GraphConv", GraphConvModel, {"in_ch": n_rois}),
        ("ChebConv", ChebModel, {"in_ch": n_rois}),
        ("Transformer", TransformerModel, {"in_ch": n_rois})
    ]
# ==========================================
# 2. 학습 및 검증 엔진
# ==========================================
# ==========================================
# 🔄 1. Loss 기록 기능이 추가된 학습 함수들로 덮어쓰기
# ==========================================

# ==========================================
# 2. 학습 및 검증 엔진
# ==========================================
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(DEVICE); optimizer.zero_grad()
        loss = criterion(model(data), data.y.squeeze())
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        total_loss += loss.item() * data.num_graphs

    return total_loss / len(loader.dataset)

# 💡 [여기가 복구된 부분입니다!] 이 함수를 꼭 넣어주세요.
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(DEVICE)
            out = model(data)
            pred = out.argmax(dim=1)
            preds.extend(pred.cpu().numpy())
            labels.extend(data.y.cpu().numpy())
    return accuracy_score(labels, preds), f1_score(labels, preds, average='macro')

def run_cv(model_cls, mkwargs, dataset, epochs):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    labels = [data.y.item() for data in dataset]
    all_models = []

    fold_accs, fold_f1s = [], []
    all_losses = [] # 💡 5개 폴드의 Loss 기록을 담을 큰 바구니

    for fold, (tr_idx, te_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        tr_data = [dataset[i] for i in tr_idx]
        te_data = [dataset[i] for i in te_idx]

        tr_labels = [d.y.item() for d in tr_data]
        class_counts = np.bincount(tr_labels)
        weights = 1. / class_counts
        sample_weights = [weights[l] for l in tr_labels]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

        tr_ld = DataLoader(tr_data, batch_size=32, sampler=sampler)
        te_ld = DataLoader(te_data, batch_size=32, shuffle=False)

        model = model_cls(**mkwargs).to(DEVICE)
        opt = Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
        criterion = nn.CrossEntropyLoss()
        sched = CosineAnnealingLR(opt, T_max=epochs)

        epoch_losses = [] # 💡 현재 폴드의 에포크별 Loss 기록장
        for ep in range(epochs):
            loss = train_epoch(model, tr_ld, opt, criterion)
            sched.step()
            epoch_losses.append(loss) # 💡 매 에포크마다 Loss 적어두기

        acc, f1 = evaluate(model, te_ld)
        # 💡 마지막 에포크의 Loss 값도 같이 출력하도록 변경
        print(f"    Fold {fold+1} -> 최종 Loss: {loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")

        all_models.append(model)
        fold_accs.append(acc)
        fold_f1s.append(f1)
        all_losses.append(epoch_losses) # 💡 큰 바구니에 현재 폴드 기록 저장

    avg_acc = np.mean(fold_accs)
    avg_f1 = np.mean(fold_f1s)
    print(f"  🏆 평균 -> Acc: {avg_acc:.4f} | F1: {avg_f1:.4f}")

    # 💡 모델뿐만 아니라 Loss 기록도 같이 반환합니다!
    return all_models, all_losses

# ==========================================
# 3. 데이터 로드 및 동적 피처 수혈 (Step 2)
# ==========================================
BASE_DIR = '/content/ASD_project_data'
target_folders = [
    "data_AAL_Adolescent_Top50_7ch",
    "data_AAL_Adult_Top50_7ch",
    "data_Schaefer_Adolescent_Top50_7ch",
    "data_Schaefer_Adult_Top50_7ch"
]

# ==========================================
# 3. [완벽 방어 적용] 메인 학습 및 평가 루프
# ==========================================
import os

# 💡 보험 1: 모델을 물리적으로 저장할 안전한 폴더를 하나 만듭니다.
SAVE_DIR = os.path.join(BASE_DIR, "saved_models")
os.makedirs(SAVE_DIR, exist_ok=True)

BEST_MODELS_DICT = {}

for folder in target_folders:
    data_path = os.path.join(BASE_DIR, folder)
    print(f"\n======================================")
    print(f" 📂 [데이터 전처리 및 훈련] {folder}")
    print(f"======================================")

    if not os.path.exists(data_path):
        for root, dirs, files in os.walk(BASE_DIR):
            if folder in dirs:
                data_path = os.path.join(root, folder)
                break

    if not os.path.exists(data_path):
         print(f"🚨 폴더를 찾을 수 없어 건너뜁니다: {folder}")
         continue

    raw_dataset = [torch.load(os.path.join(data_path, f), map_location='cpu', weights_only=False)
               for f in os.listdir(data_path) if f.startswith('sub_')]

    enriched_dataset = []
    for data in raw_dataset:
        row, col = data.edge_index
        deg = degree(row, num_nodes=data.num_nodes).view(-1, 1)
        deg = deg / (deg.max() + 1e-6)
        data.x = torch.cat([data.x, deg], dim=-1)
        enriched_dataset.append(data)

    if enriched_dataset:
        input_dim = enriched_dataset[0].x.shape[1]
        print(f"\n✅ 모델 입력 차원 감지 완료: {input_dim}차원")

        for mname, mcls, _ in get_model_registry(n_rois=input_dim):
            print(f"\n🚀 [{mname}] 모델 학습을 시작합니다...")

            # 💡 [에러 원인 완벽 수정] 튜플을 정확히 두 변수(trained_models, losses)로 나눠 받습니다.
            trained_models, losses = run_cv(mcls, {"in_ch": input_dim}, enriched_dataset, epochs=150)

            # 가장 학습이 잘 된 마지막 에포크의 모델
            best_model = trained_models[-1]

            # 💡 보험 2: 딕셔너리에 안전하게 저장 (RAM 보관용)
            BEST_MODELS_DICT[(folder, mname)] = best_model

            # 💡 보험 3: 코랩이 튕겨도 안 날아가게 물리적 파일(.pth)로 영구 저장 (하드디스크 보관용)
            save_name = f"{folder}_{mname}_best.pth"
            save_path = os.path.join(SAVE_DIR, save_name)
            torch.save(best_model.state_dict(), save_path)
            print(f"  💾 [안전 저장 완료] {save_name} 파일이 디스크에 저장되었습니다.")

print("\n🎉 모든 모델 고도화 및 150 에포크 릴레이가 완벽하게 완료되었습니다!")

In [ ]:
# ==========================================
# [Cell 4] XAI (설명 가능한 AI) 엑스레이 분석
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
import torch

def get_node_saliency(model, data, target_class=1):
    model.eval()
    data.x.requires_grad = True # 역전파 추적 시작

    out = model(data)
    score = out[0, target_class]

    model.zero_grad()
    score.backward()

    # 중요도(Saliency) 계산 및 Min-Max 정규화
    saliency = data.x.grad.abs().sum(dim=1).cpu().numpy()
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)

    return saliency

def run_xai_analysis(model, dataset, patient_idx=0, top_k=10):
    data = dataset[patient_idx].to(DEVICE)
    actual_label = data.y.item()

    model.eval()
    with torch.no_grad():
        out = model(data)
        pred = out.argmax(dim=1).item()

    print(f"\n🔍 [XAI 엑스레이 분석] 환자 인덱스: {patient_idx}")
    print(f" 🎯 실제 진단: {'ASD (자폐)' if actual_label == 1 else 'TC (정상)'}")
    print(f" 🤖 모델 예측: {'ASD (자폐)' if pred == 1 else 'TC (정상)'}")

    saliency = get_node_saliency(model, data, target_class=pred)

    top_indices = np.argsort(saliency)[-top_k:][::-1]
    top_scores = saliency[top_indices]

    print(f"\n🧠 판별에 결정적 영향을 미친 Top {top_k} 뇌 구역:")
    for rank, node_idx in enumerate(top_indices):
        print(f" {rank+1}위: 노드 [{node_idx}] (중요도: {top_scores[rank]:.4f})")

    plt.figure(figsize=(10, 5))
    plt.bar(range(top_k), top_scores, color='royalblue', edgecolor='black')
    plt.xticks(range(top_k), [f"Node {i}" for i in top_indices], rotation=45)
    plt.title(f"XAI Saliency: Top {top_k} Important Brain Regions")
    plt.ylabel("Importance Score (0~1)")
    plt.xlabel("Brain Region (Node Index)")
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# 💡 실행: AAL 청소년 데이터에서 가장 성능이 좋았던 GAT 모델로 0번 환자 분석
# (주의: 앞선 학습 루프가 끝난 뒤에 실행해야 BEST_MODELS_DICT가 존재합니다!)
try:
    best_model = BEST_MODELS_DICT[("data_AAL_Adolescent_Top50_7ch", "GAT")]
    run_xai_analysis(best_model, enriched_dataset, patient_idx=0, top_k=10)
except KeyError:
    print("🚨 아직 모델이 학습되지 않았거나 딕셔너리에 저장되지 않았습니다. 학습 셀을 먼저 실행해주세요.")

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch_geometric.utils import degree # 💡 degree 계산 도구 추가

# ==========================================
# 1. 불러올 타겟 데이터 및 모델 지정
# ==========================================
TARGET_FOLDER = "data_Schaefer_Adolescent_Top50_7ch"
TARGET_MODEL_NAME = "GAT"

# 경로 설정
SAVE_DIR = os.path.join(BASE_DIR, "saved_models")
weights_name = f"{TARGET_FOLDER}_{TARGET_MODEL_NAME}_best.pth"
weights_path = os.path.join(SAVE_DIR, weights_name)

# ==========================================
# 2. 데이터셋 로드 및 피처 수혈 (7ch -> 8ch 완벽 복구)
# ==========================================
data_path = os.path.join(BASE_DIR, TARGET_FOLDER)
raw_dataset = [torch.load(os.path.join(data_path, f), map_location='cpu', weights_only=False)
               for f in os.listdir(data_path) if f.startswith('sub_')]

# 💡 [핵심 수정] 학습 때와 똑같이 degree 피처를 추가하여 8차원으로 만들어 줍니다!
xai_dataset = []
for data in raw_dataset:
    row, col = data.edge_index
    deg = degree(row, num_nodes=data.num_nodes).view(-1, 1)
    deg = deg / (deg.max() + 1e-6)
    data.x = torch.cat([data.x, deg], dim=-1) # 7ch -> 8ch 확장
    xai_dataset.append(data)

# 입력 차원 자동 감지 (이제 정상적으로 8차원이 감지됩니다)
input_dim = xai_dataset[0].x.shape[1]

if TARGET_MODEL_NAME == "GCN": xai_model = GCNModel(in_ch=input_dim)
elif TARGET_MODEL_NAME == "GAT": xai_model = GATModel(in_ch=input_dim)
elif TARGET_MODEL_NAME == "GIN": xai_model = GINModel(in_ch=input_dim)
elif TARGET_MODEL_NAME == "GraphSAGE": xai_model = SAGEModel(in_ch=input_dim)
elif TARGET_MODEL_NAME == "GraphConv": xai_model = GraphConvModel(in_ch=input_dim)
elif TARGET_MODEL_NAME == "ChebConv": xai_model = ChebModel(in_ch=input_dim)
elif TARGET_MODEL_NAME == "Transformer": xai_model = TransformerModel(in_ch=input_dim)

# ==========================================
# 3. 물리 파일(.pth) 가중치 수혈
# ==========================================
if os.path.exists(weights_path):
    xai_model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    xai_model.to(DEVICE)
    print(f"✅ [불러오기 성공] {weights_name} 모델을 디스크에서 성공적으로 로드했습니다.")
    print(f"📊 [데이터 확인] 분석할 환자 수: {len(xai_dataset)}명 | 입력 피처 차원: {input_dim}차원")
else:
    print(f"🚨 [파일 없음] {weights_path} 경로에 모델 파일이 없습니다.")

# ==========================================
# 4. XAI 파이프라인 가동 (0번 환자 엑스레이)
# ==========================================
if os.path.exists(weights_path):
    run_xai_analysis(xai_model, xai_dataset, patient_idx=0, top_k=10)

In [ ]:
import nilearn.datasets as nds

# Load Schaefer atlas — check if 200 or 400
# TARGET_FOLDER says "Top50_7ch" with 827 subjects — likely Schaefer200
schaefer = nds.fetch_atlas_schaefer_2018(n_rois=400)
labels = [l.decode() if isinstance(l, bytes) else l 
          for l in schaefer.labels]

top_nodes = [256, 213, 396, 383, 257, 121, 370, 368, 97, 123]

print("Node → Region Name")
print("-"*60)
for n in top_nodes:
    if n < len(labels):
        print(f"  Node {n:3d} → {labels[n]}")
    else:
        print(f"  Node {n:3d} → out of range (atlas has {len(labels)} ROIs)")

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch_geometric.utils import degree # 💡 degree 계산 도구 추가

# ==========================================
# 1. 불러올 타겟 데이터 및 모델 지정
# ==========================================
TARGET_FOLDER = "data_AAL_Adolescent_Top50_7ch"
TARGET_MODEL_NAME = "GAT"

# 경로 설정
SAVE_DIR = os.path.join(BASE_DIR, "saved_models")
weights_name = f"{TARGET_FOLDER}_{TARGET_MODEL_NAME}_best.pth"
weights_path = os.path.join(SAVE_DIR, weights_name)

# ==========================================
# 2. 데이터셋 로드 및 피처 수혈 (7ch -> 8ch 완벽 복구)
# ==========================================
data_path = os.path.join(BASE_DIR, TARGET_FOLDER)
raw_dataset = [torch.load(os.path.join(data_path, f), map_location='cpu', weights_only=False)
               for f in os.listdir(data_path) if f.startswith('sub_')]

# 💡 [핵심 수정] 학습 때와 똑같이 degree 피처를 추가하여 8차원으로 만들어 줍니다!
xai_dataset = []
for data in raw_dataset:
    row, col = data.edge_index
    deg = degree(row, num_nodes=data.num_nodes).view(-1, 1)
    deg = deg / (deg.max() + 1e-6)
    data.x = torch.cat([data.x, deg], dim=-1) # 7ch -> 8ch 확장
    xai_dataset.append(data)

# 입력 차원 자동 감지 (이제 정상적으로 8차원이 감지됩니다)
input_dim = xai_dataset[0].x.shape[1]

if TARGET_MODEL_NAME == "GCN": xai_model = GCNModel(in_ch=input_dim)
elif TARGET_MODEL_NAME == "GAT": xai_model = GATModel(in_ch=input_dim)
elif TARGET_MODEL_NAME == "GIN": xai_model = GINModel(in_ch=input_dim)
elif TARGET_MODEL_NAME == "GraphSAGE": xai_model = SAGEModel(in_ch=input_dim)
elif TARGET_MODEL_NAME == "GraphConv": xai_model = GraphConvModel(in_ch=input_dim)
elif TARGET_MODEL_NAME == "ChebConv": xai_model = ChebModel(in_ch=input_dim)
elif TARGET_MODEL_NAME == "Transformer": xai_model = TransformerModel(in_ch=input_dim)

# ==========================================
# 3. 물리 파일(.pth) 가중치 수혈
# ==========================================
if os.path.exists(weights_path):
    xai_model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    xai_model.to(DEVICE)
    print(f"✅ [불러오기 성공] {weights_name} 모델을 디스크에서 성공적으로 로드했습니다.")
    print(f"📊 [데이터 확인] 분석할 환자 수: {len(xai_dataset)}명 | 입력 피처 차원: {input_dim}차원")
else:
    print(f"🚨 [파일 없음] {weights_path} 경로에 모델 파일이 없습니다.")

# ==========================================
# 4. XAI 파이프라인 가동 (0번 환자 엑스레이)
# ==========================================
if os.path.exists(weights_path):
    run_xai_analysis(xai_model, xai_dataset, patient_idx=0, top_k=10)